# set up environment

In [7]:
# @title {display-mode: "form"}

import os
import sys
from collections.abc import Callable

from google import genai

# Manual setup (leave unchanged if setup is environment-defined)

# @markdown **Which API: Vertex AI or Google AI Studio?**
GOOGLE_GENAI_USE_VERTEXAI = False  # @param {type: "boolean"}

# @markdown **Option A - Google Cloud project [+location]**
GOOGLE_CLOUD_PROJECT = ""  # @param {type: "string"}
GOOGLE_CLOUD_LOCATION = "global"  # @param {type: "string"}

# @markdown **Option B - Google AI Studio API key**
# GOOGLE_API_KEY = ""  # @param {type: "string"}
GOOGLE_API_KEY = "REMOVED"  # professor api key


def check_environment() -> bool:
    check_colab_user_authentication()
    return check_manual_setup() or check_vertex_ai() or check_colab() or check_local()


def check_manual_setup() -> bool:
    return check_define_env_vars(
        GOOGLE_GENAI_USE_VERTEXAI,
        GOOGLE_CLOUD_PROJECT.strip(),  # Might have been pasted with line return
        GOOGLE_CLOUD_LOCATION,
        GOOGLE_API_KEY,
    )


def check_vertex_ai() -> bool:
    # Workbench and Colab Enterprise
    match os.getenv("VERTEX_PRODUCT", ""):
        case "WORKBENCH_INSTANCE":
            pass
        case "COLAB_ENTERPRISE":
            if not running_in_colab_env():
                return False
        case _:
            return False

    return check_define_env_vars(
        True,
        os.getenv("GOOGLE_CLOUD_PROJECT", ""),
        os.getenv("GOOGLE_CLOUD_REGION", ""),
        "",
    )


def check_colab() -> bool:
    if not running_in_colab_env():
        return False

    # Colab Enterprise was checked before, so this is Colab only
    from google.colab import auth as colab_auth  # type: ignore

    colab_auth.authenticate_user()

    # Use Colab Secrets (🗝️ icon in left panel) to store the environment variables
    # Secrets are private, visible only to you and the notebooks that you select
    # - Vertex AI: Store your settings as secrets
    # - Google AI: Directly import your Gemini API key from the UI
    vertexai, project, location, api_key = get_vars(get_colab_secret)

    return check_define_env_vars(vertexai, project, location, api_key)


def check_local() -> bool:
    vertexai, project, location, api_key = get_vars(os.getenv)

    return check_define_env_vars(vertexai, project, location, api_key)


def running_in_colab_env() -> bool:
    # Colab or Colab Enterprise
    return "google.colab" in sys.modules


def check_colab_user_authentication() -> None:
    if running_in_colab_env():
        from google.colab import auth as colab_auth  # type: ignore

        colab_auth.authenticate_user()


def get_colab_secret(secret_name: str, default: str) -> str:
    from google.colab import userdata  # type: ignore

    try:
        return userdata.get(secret_name)
    except Exception:
        return default


def get_vars(getenv: Callable[[str, str], str]) -> tuple[bool, str, str, str]:
    # Limit getenv calls to the minimum (may trigger UI confirmation for secret access)
    vertexai_str = getenv("GOOGLE_GENAI_USE_VERTEXAI", "")
    if vertexai_str:
        vertexai = vertexai_str.lower() in ["true", "1"]
    else:
        vertexai = bool(getenv("GOOGLE_CLOUD_PROJECT", ""))

    project = getenv("GOOGLE_CLOUD_PROJECT", "") if vertexai else ""
    location = getenv("GOOGLE_CLOUD_LOCATION", "") if project else ""
    api_key = getenv("GOOGLE_API_KEY", "") if not project else ""

    return vertexai, project, location, api_key


def check_define_env_vars(
    vertexai: bool,
    project: str,
    location: str,
    api_key: str,
) -> bool:
    match (vertexai, bool(project), bool(location), bool(api_key)):
        case (True, True, _, _):
            # Vertex AI - Google Cloud project [+location]
            location = location or "global"
            define_env_vars(vertexai, project, location, "")
        case (True, False, _, True):
            # Vertex AI - API key
            define_env_vars(vertexai, "", "", api_key)
        case (False, _, _, True):
            # Google AI Studio - API key
            define_env_vars(vertexai, "", "", api_key)
        case _:
            return False

    return True


def define_env_vars(vertexai: bool, project: str, location: str, api_key: str) -> None:
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = str(vertexai)
    os.environ["GOOGLE_CLOUD_PROJECT"] = project
    os.environ["GOOGLE_CLOUD_LOCATION"] = location
    os.environ["GOOGLE_API_KEY"] = api_key


def check_configuration(client: genai.Client) -> None:
    service = "Vertex AI" if client.vertexai else "Google AI Studio"
    print(f"Using the {service} API", end="")

    if client._api_client.project:
        print(f' with project "{client._api_client.project[:7]}…"', end="")
        print(f' in location "{client._api_client.location}"')
    elif client._api_client.api_key:
        api_key = client._api_client.api_key
        print(f' with API key "{api_key[:5]}…{api_key[-5:]}"', end="")
        print(f" (in case of error, make sure it was created for {service})")

In [8]:
from google import genai

check_environment()

client = genai.Client()

In [13]:
import re
import os
import pathlib
import mimetypes
import itertools
import tenacity
from collections.abc import Callable, Iterator
from typing import Union
from datetime import timedelta

import IPython
from IPython.display import display, Markdown, HTML, Audio as IPythonAudio
import pydantic
from pandas import DataFrame, Series
from pandas.io.formats.style import Styler
from pandas.io.formats.style_render import CSSDict
from google.genai.types import (
    MediaResolution, 
    ThinkingConfig, 
    GenerateContentConfig,
    GenerateContentResponse,
    Part,
    FileData,
    FinishReason,
    VideoMetadata
    
)


# --- Configuration & Constants ---
SamplingFrameRate = float
NOT_FOUND = "?"
BGCOLOR_COLUMN = "bg_color"
YOUTUBE_URL_PREFIX = "https://www.youtube.com/watch?v="
CLOUD_STORAGE_URI_PREFIX = "gs://"

# --- Mock/Placeholder Classes (Ensure these match your actual environment) ---
# Assuming Video/Audio are Enums or Classes with a 'value' or 'path' attribute
# If these are imported from a library, you can remove these dummy definitions.
class Video:
    def __init__(self, value, name="video"):
        self.value = value
        self.name = name

class Audio:
    def __init__(self, value, name="audio"):
        self.value = value
        self.name = name

class VideoSegment:
    def __init__(self, start: timedelta, end: timedelta):
        self.start = start
        self.end = end

class Model:
    GEMINI_2_0_FLASH = "gemini-2.0-flash-exp"
    GEMINI_2_5_FLASH = "gemini-2.5-flash"
    GEMINI_2_5_PRO = "gemini-2.5-pro"
    DEFAULT = GEMINI_2_0_FLASH

# --- Prompts ---
VIDEO_TRANSCRIPTION_PROMPT = """
**Task 1 - Transcripts**
- Watch the video and listen carefully to the audio.
- Identify each unique voice using a `voice` ID (1, 2, 3, etc.).
- Transcribe the video's audio verbatim with voice diarization.
- Include the `start` timecode ({timecode_spec}) for each speech segment.

**Task 2 - Speakers**
- For each `voice` ID from Task 1, extract information about the corresponding speaker.
- Use visual and audio cues.
- If a piece of information cannot be found, use a question mark (`?`) as the value.
"""

AUDIO_TRANSCRIPTION_PROMPT = """
**Task 1 - Transcripts**
- Listen carefully to the audio file.
- Identify each unique voice using a `voice` ID (1, 2, 3, etc.).
- Transcribe the audio verbatim with voice diarization.
- Include the `start` timecode ({timecode_spec}) for each speech segment.

**Task 2 - Speakers**
- For each `voice` ID from Task 1, extract information about the corresponding speaker.
- Use audio context (introductions, addressing by name).
- If a piece of information cannot be found, use a question mark (`?`) as the value.
- Note: 'role_in_video' should be interpreted as 'role_in_conversation'.
"""

# --- Pydantic Models ---
class Transcript(pydantic.BaseModel):
    start: str
    text: str
    voice: int

class Speaker(pydantic.BaseModel):
    voice: int
    name: str
    company: str
    position: str
    role_in_video: str 

class VideoTranscription(pydantic.BaseModel):
    task1_transcripts: list[Transcript] = pydantic.Field(default_factory=list)
    task2_speakers: list[Speaker] = pydantic.Field(default_factory=list)

# --- Helper Functions ---

def convert_to_https_url_if_cloud_storage_uri(uri: str) -> str:
    if uri.startswith(CLOUD_STORAGE_URI_PREFIX):
        return f"https://storage.googleapis.com/{uri.removeprefix(CLOUD_STORAGE_URI_PREFIX)}"
    return uri

def get_video_part_metadata(
    video_segment: VideoSegment | None = None,
    fps: float | None = None,
) -> VideoMetadata:
    def offset_as_str(offset: timedelta) -> str:
        return f"{offset.total_seconds()}s"

    if video_segment:
        start_offset = offset_as_str(video_segment.start)
        end_offset = offset_as_str(video_segment.end)
    else:
        start_offset = None
        end_offset = None

    return VideoMetadata(start_offset=start_offset, end_offset=end_offset, fps=fps)

def get_video_part(
    video: Video,
    video_segment: VideoSegment | None = None,
    fps: float | None = None,
) -> Part | None:
    video_uri: str = video.value
    video_metadata = get_video_part_metadata(video_segment, fps)

    if video_uri.startswith(YOUTUBE_URL_PREFIX) or video_uri.startswith(CLOUD_STORAGE_URI_PREFIX):
        video_uri_to_use = video_uri
        
        # Placeholder for client logic (ensure 'client' is defined in your scope)
        # if not client.vertexai: ... 

        file_data = FileData(file_uri=video_uri_to_use, mime_type="video/*")
        return Part(file_data=file_data, video_metadata=video_metadata)

    elif os.path.exists(video_uri):
        print(f"Processing local file as inline_data: {video_uri}")
        try:
            path = pathlib.Path(video_uri)
            mime_type, _ = mimetypes.guess_type(path)
            if mime_type is None:
                mime_type = "video/*"
            
            file_bytes = path.read_bytes()
            inline_data_blob = {"mime_type": mime_type, "data": file_bytes}
            return Part(inline_data=inline_data_blob, video_metadata=video_metadata)
        except Exception as e:
            print(f"❌ Error reading local file {video_uri}: {e}")
            return None
    else:
        print(f"❌ Video path/URI not recognized or file not found: {video_uri}")
        return None

def get_audio_part(audio: Audio) -> Part | None:
    # Robustly get path/value
    audio_uri: str = getattr(audio, "path", getattr(audio, "value", None))
    
    if not audio_uri:
        print("❌ Audio object has no 'path' or 'value' attribute.")
        return None

    if audio_uri.startswith(CLOUD_STORAGE_URI_PREFIX) or audio_uri.startswith("http"):
        # Placeholder for client logic
        # if not client.vertexai: ...
        
        print(f"Processing remote audio: {audio_uri}")
        file_data = FileData(file_uri=audio_uri, mime_type="audio/*")
        return Part(file_data=file_data)

    elif os.path.exists(audio_uri):
        # print(f"Processing local audio file as inline_data: {audio_uri}")
        try:
            path = pathlib.Path(audio_uri)
            mime_type, _ = mimetypes.guess_type(path)
            if mime_type is None:
                mime_type = "audio/*"
            
            file_bytes = path.read_bytes()
            inline_data_blob = {"mime_type": mime_type, "data": file_bytes}
            return Part(inline_data=inline_data_blob)
        except Exception as e:
            print(f"❌ Error reading local audio file {audio_uri}: {e}")
            return None
    else:
        print(f"❌ Audio path/URI not recognized or file not found: {audio_uri}")
        return None

def should_retry_request(exception):
    # Add your specific retry logic here
    return True

def get_retrier() -> tenacity.Retrying:
    return tenacity.Retrying(
        stop=tenacity.stop_after_attempt(7),
        wait=tenacity.wait_incrementing(start=10, increment=1),
        retry=should_retry_request,
        reraise=True,
    )

def get_media_duration(media: Union[Video, Audio]) -> timedelta | None:
    # Note: 'media.name' must exist. Ensure your Audio/Video classes have a name or handle AttributeError
    if not hasattr(media, 'name'):
        return None
        
    regex = r"_PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?$"
    if not (match := re.search(regex, media.name)):
        return None
    h_str, m_str, s_str = match.groups()
    return timedelta(
        hours=int(h_str) if h_str else 0,
        minutes=int(m_str) if m_str else 0,
        seconds=int(s_str) if s_str else 0,
    )

def get_timecode_spec_for_media(model: Model, media: Union[Video, Audio]) -> str:
    timecode_spec = "MM:SS"
    # Simple check if model supports long durations
    if model in [Model.GEMINI_2_5_FLASH, Model.GEMINI_2_5_PRO]:
        duration = get_media_duration(media)
        if duration and duration >= timedelta(hours=1):
            timecode_spec = "MM:SS or H:MM:SS"
    return timecode_spec

# --- Configuration Helpers ---

def get_thinking_config(model: Model) -> ThinkingConfig | None:
    if model == Model.GEMINI_2_5_FLASH:
        return ThinkingConfig(thinking_budget=0, include_thoughts=False)
    elif model == Model.GEMINI_2_5_PRO:
        return ThinkingConfig(thinking_budget=128, include_thoughts=False)
    return None

def get_generate_content_config(model: Model, media_resolution: MediaResolution | None) -> GenerateContentConfig:
    thinking_config = get_thinking_config(model)
    # Ensure DEFAULT_CONFIG is defined in your global scope
    return GenerateContentConfig(
        temperature=0.0, # defaulting if DEFAULT_CONFIG missing
        top_p=0.95,
        seed=42,
        response_mime_type="application/json",
        response_schema=VideoTranscription,
        media_resolution=media_resolution,
        thinking_config=thinking_config,
    )

def get_sampling_frame_rate_for_video(video: Video) -> SamplingFrameRate | None:
    return None

def get_video_transcription_from_response(response: GenerateContentResponse) -> VideoTranscription:
    if not isinstance(response.parsed, VideoTranscription):
        print("❌ Could not parse the JSON response")
        return VideoTranscription()
    return response.parsed

# --- Main Transcription Logic ---

def get_media_transcription(
    media: Union[Video, Audio],
    segment: Union[VideoSegment, None] = None,
    fps: float | None = None,
    prompt: str | None = None,
    model: Model | None = None,
) -> tuple[VideoTranscription, dict[str, int] | None]:
    model = model or Model.DEFAULT
    model_id = model # assuming Model enum values are strings

    if isinstance(media, Video):
        fps = fps or get_sampling_frame_rate_for_video(media)
        media_part = get_video_part(media, segment, fps)
        default_prompt = VIDEO_TRANSCRIPTION_PROMPT
        media_res = None # Placeholder logic
    elif isinstance(media, Audio):
        media_part = get_audio_part(media)
        default_prompt = AUDIO_TRANSCRIPTION_PROMPT
        media_res = None
    else:
        return VideoTranscription()

    if not media_part:
        print("❌ Failed to create media part.")
        return VideoTranscription()

    if prompt is None:
        timecode_spec = get_timecode_spec_for_media(model, media)
        prompt = default_prompt.format(timecode_spec=timecode_spec)

    contents = [media_part, prompt.strip()]
    config = get_generate_content_config(model, media_res)

    # print(f" Processing: {getattr(media, 'name', 'media')} ".center(80, "-"))
    
    response = None
    # Ensure 'client' is initialized in your environment
    # for attempt in get_retrier():
    #     with attempt:
            # client.models.generate_content(...) call
            # Mocking response for syntax correctness if client is missing
            # pass 
            
            # Uncomment below when running with actual client
            # print("calling client")
    response = client.models.generate_content(
        model=model_id,
        contents=contents,
        config=config,
    )
    # print(response)
    token_dict = display_response_info(response)
    
    # Returning empty if no response (due to missing client)
    if response is None:
         print("⚠️ No response generated (Client not initialized in this snippet)")
         return VideoTranscription()

    return get_video_transcription_from_response(response), token_dict

# --- Display Logic ---


def display_markdown(markdown: str) -> None:
    IPython.display.display(IPython.display.Markdown(markdown))

def display_video(video: Video) -> None:
    video_url = convert_to_https_url_if_cloud_storage_uri(video.value)

    if not video_url.startswith("https://") and not video_url.startswith("gs://"):
        print(f"Displaying local video file: {video_url}")
        ipython_video = IPython.display.Video(video_url, width=600)
        display_markdown(f"### Video (local file)")
    elif video_url.startswith(YOUTUBE_URL_PREFIX):
        youtube_id = video_url.removeprefix(YOUTUBE_URL_PREFIX)
        ipython_video = IPython.display.YouTubeVideo(youtube_id, width=600)
        display_markdown(f"### Video ([source]({video_url}))")
    else:
        ipython_video = IPython.display.Video(video_url, width=600)
        display_markdown(f"### Video ([source]({video_url}))")

    IPython.display.display(ipython_video)

def display_audio(media: Union[Audio, str]) -> None:
    """
    Displays an audio player. Accepts either an Audio object or a string path.
    """
    # 1. Extract the actual path string
    if isinstance(media, str):
        audio_path = media
    else:
        # Try finding 'path' or 'value' attribute on the object
        audio_path = getattr(media, "path", getattr(media, "value", None))
    
    if not audio_path:
        print("❌ Cannot display audio: Invalid media object or path.")
        return

    # 2. Check if the file actually exists locally
    if os.path.exists(audio_path):
        display(IPythonAudio(filename=audio_path))
    else:
        # 3. Handle Cloud URIs or missing files
        if audio_path.startswith("gs://"):
            display(Markdown(f"**Note:** Audio is stored on Cloud Storage (`{audio_path}`) and cannot be played directly."))
        else:
            display(Markdown(f"⚠️ **File not found:** `{audio_path}`. Cannot play audio."))

# --- Table Display Functions (Unchanged logic, compacted for brevity) ---
def yield_known_speaker_color() -> Iterator[str]:
    PAL_40 = ("#669DF6", "#EE675C", "#FCC934", "#5BB974")
    return itertools.cycle(PAL_40) # Simplified for brevity

def yield_unknown_speaker_color() -> Iterator[str]:
    GRAYS = ["#80868B", "#9AA0A6"]
    return itertools.cycle(GRAYS)

def get_color_for_voice_mapping(speakers: list[Speaker]) -> dict[int, str]:
    known = yield_known_speaker_color()
    unknown = yield_unknown_speaker_color()
    mapping = {}
    for s in speakers:
        mapping[s.voice] = next(known) if s.name != NOT_FOUND else next(unknown)
    return mapping

import itertools
from collections.abc import Callable, Iterator

from pandas import DataFrame, Series
from pandas.io.formats.style import Styler
from pandas.io.formats.style_render import CSSDict

BGCOLOR_COLUMN = "bg_color"  # Hidden column to store row background colors


def yield_known_speaker_color() -> Iterator[str]:
    PAL_40 = ("#669DF6", "#EE675C", "#FCC934", "#5BB974")
    PAL_30 = ("#8AB4F8", "#F28B82", "#FDD663", "#81C995")
    PAL_20 = ("#AECBFA", "#F6AEA9", "#FDE293", "#A8DAB5")
    PAL_10 = ("#D2E3FC", "#FAD2CF", "#FEEFC3", "#CEEAD6")
    PAL_05 = ("#E8F0FE", "#FCE8E6", "#FEF7E0", "#E6F4EA")
    return itertools.cycle([*PAL_40, *PAL_30, *PAL_20, *PAL_10, *PAL_05])


def yield_unknown_speaker_color() -> Iterator[str]:
    GRAYS = ["#80868B", "#9AA0A6", "#BDC1C6", "#DADCE0", "#E8EAED", "#F1F3F4"]
    return itertools.cycle(GRAYS)


def get_color_for_voice_mapping(speakers: list[Speaker]) -> dict[int, str]:
    known_speaker_color = yield_known_speaker_color()
    unknown_speaker_color = yield_unknown_speaker_color()

    mapping: dict[int, str] = {}
    for speaker in speakers:
        if speaker.name != NOT_FOUND:
            color = next(known_speaker_color)
        else:
            color = next(unknown_speaker_color)
        mapping[speaker.voice] = color

    return mapping


def get_table_styler(df: DataFrame) -> Styler:
    def join_styles(styles: list[str]) -> str:
        return ";".join(styles)

    table_css = [
        "color: #202124",
        "background-color: #BDC1C6",
        "border: 0",
        "border-radius: 0.5rem",
        "border-spacing: 0px",
        "outline: 0.5rem solid #BDC1C6",
        "margin: 1rem 0.5rem",
    ]
    th_css = ["background-color: #E8EAED"]
    th_td_css = ["text-align:left", "padding: 0.25rem 1rem"]
    table_styles = [
        CSSDict(selector="", props=join_styles(table_css)),
        CSSDict(selector="th", props=join_styles(th_css)),
        CSSDict(selector="th,td", props=join_styles(th_td_css)),
    ]

    return df.style.set_table_styles(table_styles).hide()


def change_row_bgcolor(row: Series) -> list[str]:
    style = f"background-color:{row[BGCOLOR_COLUMN]}"
    return [style] * len(row)


def display_table(yield_rows: Callable[[], Iterator[list[str]]]) -> None:
    data = yield_rows()
    df = DataFrame(columns=next(data), data=data)
    styler = get_table_styler(df)
    styler.apply(change_row_bgcolor, axis=1)
    styler.hide([BGCOLOR_COLUMN], axis="columns")

    html = styler.to_html()
    IPython.display.display(IPython.display.HTML(html))


def display_speakers(transcription: VideoTranscription) -> None:
    def sanitize_field(s: str, symbol_if_unknown: str) -> str:
        return symbol_if_unknown if s == NOT_FOUND else s

    def yield_rows() -> Iterator[list[str]]:
        yield ["voice", "name", "company", "position", "role_in_video", BGCOLOR_COLUMN]

        color_for_voice = get_color_for_voice_mapping(transcription.task2_speakers)
        for speaker in transcription.task2_speakers:
            yield [
                str(speaker.voice),
                sanitize_field(speaker.name, NOT_FOUND),
                sanitize_field(speaker.company, NOT_FOUND),
                sanitize_field(speaker.position, NOT_FOUND),
                sanitize_field(speaker.role_in_video, NOT_FOUND),
                color_for_voice.get(speaker.voice, "red"),
            ]

    display_markdown(f"### Speakers ({len(transcription.task2_speakers)})")
    display_table(yield_rows)


def display_transcripts(transcription: VideoTranscription) -> None:
    def yield_rows() -> Iterator[list[str]]:
        yield ["start", "speaker", "transcript", BGCOLOR_COLUMN]

        color_for_voice = get_color_for_voice_mapping(transcription.task2_speakers)
        speaker_for_voice = {
            speaker.voice: speaker for speaker in transcription.task2_speakers
        }
        previous_voice = None
        for transcript in transcription.task1_transcripts:
            current_voice = transcript.voice
            speaker_label = ""
            if speaker := speaker_for_voice.get(current_voice):
                if speaker.name != NOT_FOUND:
                    speaker_label = speaker.name
                elif speaker.position != NOT_FOUND:
                    speaker_label = f"[voice {current_voice}][{speaker.position}]"
                elif speaker.role_in_video != NOT_FOUND:
                    speaker_label = f"[voice {current_voice}][{speaker.role_in_video}]"
            if not speaker_label:
                speaker_label = f"[voice {current_voice}]"
            yield [
                transcript.start,
                speaker_label if current_voice != previous_voice else '"',
                transcript.text,
                color_for_voice.get(current_voice, "red"),
            ]
            previous_voice = current_voice

    display_markdown(f"### Transcripts ({len(transcription.task1_transcripts)})")
    display_table(yield_rows)

def display_response_info(response: GenerateContentResponse) -> dict[str, int] | None:
    input_tokens = 0
    output_tokens = 0
    thought_tokens = 0
    if usage_metadata := response.usage_metadata:
        if usage_metadata.prompt_token_count:
            # print(f"Input tokens   : {usage_metadata.prompt_token_count:9,d}")
            input_tokens = usage_metadata.prompt_token_count
        if usage_metadata.candidates_token_count:
            # print(f"Output tokens  : {usage_metadata.candidates_token_count:9,d}")
            output_tokens = usage_metadata.candidates_token_count
        if usage_metadata.thoughts_token_count:
            # print(f"Thoughts tokens: {usage_metadata.thoughts_token_count:9,d}")
            thought_tokens = usage_metadata.thoughts_token_count

    if not response.candidates:
        print("❌ No `response.candidates`")
        return
    if (finish_reason := response.candidates[0].finish_reason) != FinishReason.STOP:
        print(f"❌ {finish_reason = }")
    if not response.text:
        print("❌ No `response.text`")
        return
    return {"input token": input_tokens, "output token": output_tokens, "thought token": thought_tokens}

# --- Main Entry Point ---

def transcribe_media(
    media: Union[Video, Audio],
    segment: Union[VideoSegment, None] = None,
    fps: float | None = None,
    prompt: str | None = None,
    model: Model | None = None,
) -> tuple[VideoTranscription, dict[str, int] | None]:
    
    # 1. Display Player
    # if isinstance(media, Video):
    #     display_video(media)
    # elif isinstance(media, Audio):
    #     display_audio(media) # Now works because display_audio handles the object
        
    # 2. Transcribe
    transcription, tokens_dict = get_media_transcription(media, segment, fps, prompt, model)
    
    # 3. Display Results
    # display_speakers(transcription) 
    # display_transcripts(transcription)
    
    return transcription, tokens_dict

In [14]:
import csv
from typing import Iterator
# --- MODIFIED FUNCTION: Save to Plain Text (.txt) ---
def save_transcription_to_txt(filename, transcription: VideoTranscription):
    """
    Saves the dialogue with speaker and timing information to a plain .txt file.
    """
    # print(f"Generating detailed text file: {filename}...")

    # Get speaker mapping for consistent labeling
    all_transcripts = transcription.task1_transcripts
    speaker_for_voice = {speaker.voice: speaker for speaker in transcription.task2_speakers}

    with open(filename, "w", encoding="utf-8") as f:
        f.write("--- Detailed Conversation Log ---\n\n")

        for transcript in all_transcripts:
            current_voice = transcript.voice
            
            speaker_label = ""
            voice_id = f" (Voice {current_voice})"
            
            if speaker := speaker_for_voice.get(current_voice):
                if speaker.name != NOT_FOUND:
                    speaker_label = speaker.name
                elif speaker.position != NOT_FOUND:
                    # Specific formatting to include voice ID for officers as requested
                    if "officer" in speaker.position.lower():
                        speaker_label = f"{speaker.position}{voice_id}"
                    else:
                        speaker_label = f"[{speaker.position}]"
                elif speaker.role_in_video != NOT_FOUND:
                    speaker_label = f"[{speaker.role_in_video}]"
            
            if not speaker_label:
                speaker_label = f"[Voice {current_voice}]"

            # Final line format: [00:00] Speaker Label: Transcript text
            line = f"[{transcript.start}] {speaker_label}: {transcript.text}\n"
            f.write(line)

    # print(f"Successfully saved detailed dialogue to {filename}")

# --- NEW FUNCTION: Save to CSV (.csv) ---
def save_transcription_to_csv(filename, transcription: VideoTranscription):
    """
    Saves the structured dialogue data (start, speaker, transcript) to a .csv file.
    
    Note: For CSV, the full speaker label is used for every line (no '""') 
    to maintain data integrity for parsing.
    """
    # print(f"Generating CSV file: {filename}...")

    # Define the columns for the CSV
    csv_rows = [["start_time", "speaker_label", "transcript"]]
    
    # Get speaker mapping for consistent labeling in CSV
    all_transcripts = transcription.task1_transcripts
    speaker_for_voice = {speaker.voice: speaker for speaker in transcription.task2_speakers}

    for transcript in all_transcripts:
        current_voice = transcript.voice
        
        # Re-derive the full speaker label (no '""' for redundancy)
        speaker_label = ""
        if speaker := speaker_for_voice.get(current_voice):
            if speaker.name != NOT_FOUND:
                speaker_label = speaker.name
            elif speaker.position != NOT_FOUND:
                speaker_label = f"[{speaker.position}]"
            elif speaker.role_in_video != NOT_FOUND:
                speaker_label = f"[{speaker.role_in_video}]"
        if not speaker_label:
            speaker_label = f"[Voice {current_voice}]"

        csv_rows.append([
            transcript.start,
            speaker_label,
            transcript.text
        ])
        
    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        csv_writer = csv.writer(csvfile)
        csv_writer.writerows(csv_rows)

    # print(f"Successfully saved structured data to {filename}")
    
def save_transcription_to_markdown(filename: str = "conversation_transcript.md"):
    """
    Saves the transcription data generated by yield_rows into a Markdown file.
    """
    rows = list(yield_rows())
    
    # 1. Prepare for Markdown Table format
    header = rows[0]
    data_rows = rows[1:]

    # Start writing to the file
    with open(filename, "w", encoding="utf-8") as f:
        f.write("# 🎙️ Video Conversation Transcript\n\n")
        f.write("## Dialogue Table\n\n")
        
        # Write the header row
        f.write("| " + " | ".join(header[:-1]) + " |\n") # Exclude BG_COLOR from table
        
        # Write the separator row
        f.write("|" + "---|" * (len(header) - 1) + "\n")
        
        # Write the data rows
        for row in data_rows:
            # Escape pipes within the transcript text for table compatibility
            transcript_text = row[2].replace("|", r"\|")
            f.write(f"| {row[0]} | {row[1]} | {transcript_text} |\n")

        f.write("\n---\n\n")
        f.write("## Continuous Dialogue\n\n")

        # 2. Prepare and write the continuous dialogue block
        for row in data_rows:
            speaker_label = row[1]
            transcript_text = row[2]
            start_time = row[0]
            
            # Format as a Blockquote/list for cleaner dialogue flow
            if speaker_label != '"':
                f.write(f"\n**{speaker_label}** *({start_time}):*\n")
            
            f.write(f"> {transcript_text}\n")


# # --- Execution ---
# if __name__ == "__main__":
#     # 1. Generate the plain text file
#     txt_filename = f"{output_dir}/txt/{file}.txt"
#     save_transcription_to_txt(txt_filename)
    
#     # 2. Generate the CSV file
#     csv_filename = f"{output_dir}/csv/{file}.csv"
#     save_transcription_to_csv(csv_filename)
    
    # 3. Generate the Markdown file
    # markdown_filename = f"{output_dir}file_{file}.md"
    # save_transcription_to_markdown(markdown_filename)

In [6]:
# import os
# import csv
# import time

# # --- Setup Paths ---
# # Ensure you defined output_dir earlier, or set it here based on your request:
# # output_dir = "../output/gemini_2.5_flash_output" 

# # Create the object
# my_audio = Audio(value=audio_path)

# # 1. Start the timer
# start_time = time.time()

# # 2. Run the transcription
# # (Assuming transcribe_media is defined elsewhere)
# transcription, tokens_dict = transcribe_media(
#     media=my_audio,
#     model=Model.GEMINI_2_5_FLASH,
# )

# # 3. Stop the timer
# end_time = time.time()

# # 4. Calculate duration
# duration = end_time - start_time

# # --- Save TXT and CSV Transcripts ---
# # Ensure subdirectories exist
# os.makedirs(os.path.join(output_dir, "txt"), exist_ok=True)
# os.makedirs(os.path.join(output_dir, "csv"), exist_ok=True)

# txt_filename = os.path.join(output_dir, "txt", f"{file}.txt")
# save_transcription_to_txt(txt_filename, transcription)

# csv_filename = os.path.join(output_dir, "csv", f"{file}.csv")
# save_transcription_to_csv(csv_filename, transcription)

# # --- Save Log File ---
# log_filename = os.path.join(output_dir, "log.csv")

# # Check if file exists (to decide if we need to write headers)
# file_exists = os.path.isfile(log_filename)

# with open(log_filename, 'a', newline='', encoding='utf-8') as csvfile:
#     csv_writer = csv.writer(csvfile)
    
#     # Write Header ONLY if the file is new
#     if not file_exists:
#         headers = ["id", "input_token", "output_token", "thought_token", "execution_time_sec", "model_name"]
#         csv_writer.writerow(headers)
    
#     # Write Data Row
#     row_data = [
#         file, 
#         tokens_dict.get("input token", 0), 
#         tokens_dict.get("output token", 0), 
#         tokens_dict.get("thought token", 0), 
#         f"{duration:.2f}", 
#         "gemini_2.5_flash"  # Corrected spelling from 'gemeni'
#     ]
#     csv_writer.writerow(row_data)

# print(f"✅ Log updated: {log_filename}")
# print(f"⏱️ Time to transcribe: {duration:.2f} seconds")

# helper function for download youtube video and convert it to audio and save the file

In [17]:
import os
import pandas as pd
import yt_dlp
import shutil
import time
from tqdm import tqdm
# --- Configuration ---
# ROOT_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) # Points to youtube_dialogue_extractor/
# DATA_FILE = os.path.join(ROOT_DIR, 'data', 'youtube_video_manifest.csv')
# DOWNLOAD_DIR = os.path.join(ROOT_DIR, 'temp_downloads')




def download_audio(youtube_url, file_id, DOWNLOAD_DIR):
    """
    Downloads audio from YouTube URL and saves it as {file_id}.mp3
    Returns the path to the downloaded file.
    """
    # Ensure download directory exists
    if not os.path.exists(DOWNLOAD_DIR):
        os.makedirs(DOWNLOAD_DIR)

    output_template = os.path.join(DOWNLOAD_DIR, f"{file_id}.%(ext)s")
    
    # yt-dlp options for best audio
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': output_template,
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'quiet': True,
        'no_warnings': True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            # print(f"⬇️  Downloading: {youtube_url}")
            ydl.download([youtube_url])
            return os.path.join(DOWNLOAD_DIR, f"{file_id}.mp3")
    except Exception as e:
        print(f"❌ Error downloading {youtube_url}: {e}")
        return None

def process_audio(audio_path, file_id, output_dir):
    """
    PLACEHOLDER: This is where you run your Whisper/Pyannote pipeline.
    """
    if not audio_path or not os.path.exists(audio_path):
        return

    # print(f"⚙️  Processing audio for ID: {file_id}...")
    
    # Create the object
    my_audio = Audio(value=audio_path)

    # 1. Start the timer
    start_time = time.time()

    # 2. Run the transcription
    # (Assuming transcribe_media is defined elsewhere)
    transcription, tokens_dict = transcribe_media(
        media=my_audio,
        model=Model.GEMINI_2_5_FLASH,
    )

    # 3. Stop the timer
    end_time = time.time()

    # 4. Calculate duration
    duration = end_time - start_time

    # --- Save TXT and CSV Transcripts ---
    # Ensure subdirectories exist
    os.makedirs(os.path.join(output_dir, "txt"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "csv"), exist_ok=True)

    txt_filename = os.path.join(output_dir, "txt", f"{file_id}.txt")
    save_transcription_to_txt(txt_filename, transcription)

    csv_filename = os.path.join(output_dir, "csv", f"{file_id}.csv")
    save_transcription_to_csv(csv_filename, transcription)

    # --- Save Log File ---
    log_filename = os.path.join(output_dir, "log.csv")

    # Check if file exists (to decide if we need to write headers)
    file_exists = os.path.isfile(log_filename)

    with open(log_filename, 'a', newline='', encoding='utf-8') as csvfile:
        csv_writer = csv.writer(csvfile)
        
        # Write Header ONLY if the file is new
        if not file_exists:
            headers = ["id", "input_token", "output_token", "thought_token", "execution_time_sec", "model_name"]
            csv_writer.writerow(headers)
        
        # Write Data Row
        row_data = [
            file_id, 
            tokens_dict.get("input token", 0), 
            tokens_dict.get("output token", 0), 
            tokens_dict.get("thought token", 0), 
            f"{duration:.2f}", 
            "gemini_2.5_flash"  # Corrected spelling from 'gemeni'
        ]
        csv_writer.writerow(row_data)
        
    # print(f"✅ Processing complete for {file_id}")

def delete_file(file_path):
    """
    Deletes the file to save space.
    """
    if file_path and os.path.exists(file_path):
        try:
            os.remove(file_path)
            # print(f"🗑️  Deleted temp file: {file_path}")
        except OSError as e:
            print(f"⚠️  Error deleting file: {e}")

# def main():
#     current_dir = os.getcwd()
#     if 'scripts' in current_dir:
#         ROOT_DIR = os.path.dirname(current_dir)
#     else:
#         ROOT_DIR = current_dir
#     DATA_FILE = os.path.join(ROOT_DIR, 'data', 'youtube_video_manifest.csv')
#     DOWNLOAD_DIR = os.path.join(ROOT_DIR, 'temp_downloads')
#     OUTPUT_DIR = os.path.join(ROOT_DIR, 'output')
#     GEMENI_2_5_FLASH_OUTPUT = os.path.join(OUTPUT_DIR, 'gemeni_2.5_flash_output')
    
    
#     if not os.path.exists(DATA_FILE):
#         print(f"Error: Could not find {DATA_FILE}")
#         return

#     df = pd.read_csv(DATA_FILE)
    
#     process_queue = df.head(1)

#     # 2. Iterate through videos
#     # We pass 'total=process_queue.shape[0]' so tqdm knows when it hits 100%
#     for index, row in tqdm(process_queue.iterrows(), total=process_queue.shape[0], desc="Processing Videos"):
#         video_id = row['id']
#         url = row['URL']
        
#         # print(f"\n--- Starting Job: {video_id} ---")
        
#         # Step A: Download
#         audio_path = download_audio(url, video_id, DOWNLOAD_DIR)
        
#         # Step B: Process (Diarization/Extraction)
#         if audio_path:
#             process_audio(audio_path, video_id, GEMENI_2_5_FLASH_OUTPUT)
            
#             # Step C: Cleanup
#             delete_file(audio_path)
    
#     # Optional: Remove the empty temp folder at the end
#     # shutil.rmtree(DOWNLOAD_DIR) 

# if __name__ == "__main__":
#     main()

# Execution pipeline

In [ ]:
# download youtube video
# convert transcript to text with speaker labels and timestamps
# save as .txt and .csv files
# delete the youtube video file
# save the token needed for particular model model name <--> id mapping <--> input token <--> time need to get the answer

In [8]:
# import os
# import pandas as pd
# import yt_dlp
# import shutil
# import time
# import csv
# from tqdm import tqdm

# # --- Main Execution ---

# def main():
#     # 1. Setup Paths
#     current_dir = os.getcwd()
#     if 'scripts' in current_dir:
#         ROOT_DIR = os.path.dirname(current_dir)
#     else:
#         ROOT_DIR = current_dir
        
#     DATA_FILE = os.path.join(ROOT_DIR, 'data', 'youtube_video_manifest.csv') # Renamed to match our previous discussion
#     DOWNLOAD_DIR = os.path.join(ROOT_DIR, 'temp_downloads')
#     OUTPUT_DIR = os.path.join(ROOT_DIR, 'output')
#     GEMINI_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'gemini_2.5_flash_output')
#     LOG_FILE = os.path.join(GEMINI_OUTPUT_DIR, 'log.csv')
    
#     # 2. Load Manifest
#     if not os.path.exists(DATA_FILE):
#         print(f"❌ Error: Could not find manifest at {DATA_FILE}")
#         return

#     df = pd.read_csv(DATA_FILE)
#     total_videos = len(df)
    
#     # 3. RESUME LOGIC: Check which videos are already done
#     processed_ids = set()
#     if os.path.exists(LOG_FILE):
#         try:
#             log_df = pd.read_csv(LOG_FILE)
#             if 'id' in log_df.columns:
#                 processed_ids = set(log_df['id'].astype(str)) # Convert to string to match df
#                 print(f"🔄 Resuming... Found {len(processed_ids)} videos already processed.")
#         except Exception as e:
#             print(f"⚠️ Could not read log file, starting from scratch. Error: {e}")

#     # Filter the dataframe to only include videos NOT in processed_ids
#     # Convert df['id'] to string to ensure matching works
#     df['id'] = df['id'].astype(str)
#     remaining_df = df[~df['id'].isin(processed_ids)]
    
#     if remaining_df.empty:
#         print("✅ All videos have been processed! Nothing to do.")
#         return

#     print(f"🚀 Starting processing on {len(remaining_df)} remaining videos (out of {total_videos} total).")

#     # 4. Main Loop with Error Handling
#     # We iterate through remaining_df, not the whole df
#     for index, row in tqdm(remaining_df.iterrows(), total=remaining_df.shape[0], desc="Processing"):
#         video_id = row['id']
#         url = row['URL'] # Ensure this matches your CSV column name
#         duration = row.get('minutes', 0)  # Optional: get duration if available
#         if duration > 45:
#             # --- Save Log File ---
#             log_filename = os.path.join(GEMINI_OUTPUT_DIR, "log.csv")

#             # Check if file exists (to decide if we need to write headers)
#             file_exists = os.path.isfile(log_filename)

#             with open(log_filename, 'a', newline='', encoding='utf-8') as csvfile:
#                 csv_writer = csv.writer(csvfile)
                
#                 # Write Header ONLY if the file is new
#                 if not file_exists:
#                     headers = ["id", "input_token", "output_token", "thought_token", "execution_time_sec", "model_name"]
#                     csv_writer.writerow(headers)
                
#                 # Write Data Row
#                 row_data = [
#                     video_id, 
#                     0, 
#                     0, 
#                     0, 
#                     0, 
#                     "gemini_2.5_flash"  # Corrected spelling from 'gemeni'
#                 ]
#                 csv_writer.writerow(row_data)
#             continue  # Skip long videos
        
#         # Define audio_path here so it's available in the finally block if needed
#         audio_path = None 

#         try:
#             # Step A: Download
#             audio_path = download_audio(url, video_id, DOWNLOAD_DIR)
            
#             # Step B: Process
#             if audio_path:
#                 process_audio(audio_path, video_id, GEMINI_OUTPUT_DIR)
            
#             # Step C: Cleanup (Happy path)
#             delete_file(audio_path)

#         except KeyboardInterrupt:
#             print(f"\n🛑 Process interrupted by user at ID: {video_id}")
#             if audio_path: delete_file(audio_path) # Clean up current file
#             break
            
#         except Exception as e:
#             # CRITICAL ERROR BLOCK
#             print(f"\n\n❌CRITICAL ERROR ENCOUNTERED")
#             print(f"--------------------------")
#             print(f"Stopped at Video ID: {video_id}")
#             print(f"Error Details: {str(e)}")
#             print(f"--------------------------")
#             print(f"To resume, simply fix the issue and re-run the script.")
            
#             # Cleanup the failed file so it doesn't rot in the temp folder
#             if audio_path: delete_file(audio_path)
            
#             # Stop execution completely
#             break

# if __name__ == "__main__":
#     main()

In [ ]:
import os
import pandas as pd
import yt_dlp
import shutil
import time
import csv
import json  # Added json import
from tqdm import tqdm

# --- Main Execution ---

def load_status_json(file_path):
    """Helper to load the status JSON safely."""
    if not os.path.exists(file_path):
        return {"processed": [], "ignore": []}
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            # Ensure required keys exist
            if "processed" not in data: data["processed"] = []
            if "ignore" not in data: data["ignore"] = []
            return data
    except Exception as e:
        print(f"⚠️ Could not read JSON status file. Starting with empty lists. Error: {e}")
        return {"processed": [], "ignore": []}

def mark_as_processed(file_path, video_id):
    """Helper to add an ID to the 'processed' list in the JSON file."""
    try:
        data = load_status_json(file_path)
        
        # Avoid duplicates
        if video_id not in data["processed"]:
            data["processed"].append(video_id)
            
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=4)
    except Exception as e:
        print(f"⚠️ Failed to update JSON status file: {e}")

def main():
    # 1. Setup Paths
    current_dir = os.getcwd()
    if 'scripts' in current_dir:
        ROOT_DIR = os.path.dirname(current_dir)
    else:
        ROOT_DIR = current_dir
        
    DATA_FILE = os.path.join(ROOT_DIR, 'data', 'youtube_video_manifest.csv')
    DOWNLOAD_DIR = os.path.join(ROOT_DIR, 'temp_downloads')
    OUTPUT_DIR = os.path.join(ROOT_DIR, 'output_rest')
    GEMINI_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'gemini_2.5_flash_output')
    
    # New JSON Status File Path
    STATUS_FILE = os.path.join(GEMINI_OUTPUT_DIR, 'process_status.json')
    
    # Ensure output directory exists for the json file
    os.makedirs(GEMINI_OUTPUT_DIR, exist_ok=True)

    # 2. Load Manifest
    if not os.path.exists(DATA_FILE):
        print(f"❌ Error: Could not find manifest at {DATA_FILE}")
        return

    df = pd.read_csv(DATA_FILE)
    total_videos = len(df)
    
    # 3. RESUME LOGIC (UPDATED): Check JSON for processed/ignore IDs
    status_data = load_status_json(STATUS_FILE)
    
    # Convert lists to sets for faster lookup and ensure strings
    processed_ids = set(str(x) for x in status_data["unable_to_process"])
    ignore_ids = set(str(x) for x in status_data["ignore"])
    
    # Combine both lists to find what to skip
    ids_to_skip = processed_ids
    
    print(f"🔄 Resuming... Found {len(processed_ids)} processed and {len(ignore_ids)} ignored.")

    # Filter the dataframe
    df['id'] = df['id'].astype(str)
    remaining_df = df[df['id'].isin(ids_to_skip)]
    
    if remaining_df.empty:
        print("✅ All videos have been processed or ignored! Nothing to do.")
        return

    print(f"🚀 Starting processing on {len(remaining_df)} remaining videos (out of {total_videos} total).")

    # 4. Main Loop with Error Handling
    for index, row in tqdm(remaining_df.iterrows(), total=remaining_df.shape[0], desc="Processing"):
        video_id = row['id']
        url = row['URL']
        duration = row.get('minutes', 0)
        print(video_id)
        
        # Check for long videos
        if duration > 45:
            # Note: You might want to also add this to the "ignore" list in JSON automatically
            # so you don't check it again, but I will keep your original CSV logging logic here.
            log_filename = os.path.join(GEMINI_OUTPUT_DIR, "log.csv")
            file_exists = os.path.isfile(log_filename)

            with open(log_filename, 'a', newline='', encoding='utf-8') as csvfile:
                csv_writer = csv.writer(csvfile)
                if not file_exists:
                    headers = ["id", "input_token", "output_token", "thought_token", "execution_time_sec", "model_name"]
                    csv_writer.writerow(headers)
                row_data = [video_id, 0, 0, 0, 0, "gemini_2.5_flash"]
                csv_writer.writerow(row_data)
            
            # OPTIONAL: Uncomment the line below if you want to permanently ignore >45m videos in JSON too
            mark_as_processed(STATUS_FILE, video_id) 
            continue 
        
        audio_path = None 

        try:
            # Step A: Download
            # Ensure download_audio is defined in your full script
            audio_path = download_audio(url, video_id, DOWNLOAD_DIR)
            
            # Step B: Process
            if audio_path:
                # Ensure process_audio is defined in your full script
                process_audio(audio_path, video_id, GEMINI_OUTPUT_DIR)
                
                # --- NEW: Update JSON Status on Success ---
                mark_as_processed(STATUS_FILE, video_id)
            
            # Step C: Cleanup
            # Ensure delete_file is defined in your full script
            delete_file(audio_path)

        except KeyboardInterrupt:
            print(f"\n🛑 Process interrupted by user at ID: {video_id}")
            if audio_path: delete_file(audio_path)
            break
            
        except Exception as e:
            print(f"\n\n❌CRITICAL ERROR ENCOUNTERED")
            print(f"--------------------------")
            print(f"Stopped at Video ID: {video_id}")
            print(f"Error Details: {str(e)}")
            print(f"--------------------------")
            if audio_path: delete_file(audio_path)
            break

if __name__ == "__main__":
    main()

🔄 Resuming... Found 304 processed and 53 ignored.
🚀 Starting processing on 304 remaining videos (out of 1261 total).


Processing:   0%|          | 0/304 [00:00<?, ?it/s]

189
198
199


Processing:   1%|          | 3/304 [02:25<4:03:36, 48.56s/it]

201


...

Processing:   3%|▎         | 8/304 [27:28<20:07:57, 244.86s/it]

219


Processing:   3%|▎         | 8/304 [36:13<22:20:07, 271.65s/it]

❌ finish_reason = <FinishReason.RECITATION: 'RECITATION'>
❌ No `response.text`
❌ Could not parse the JSON response


❌CRITICAL ERROR ENCOUNTERED
--------------------------
Stopped at Video ID: 219
Error Details: 'NoneType' object has no attribute 'get'
--------------------------


Bad pipe message: %s [b'\xb4\xaf#\x1f\x9b\xe1g\x0b\xea\x8c\xa8\xa8\xda\x97\xa4\xb0\xae\x86\x00\x01|\x00\x00\x00\x01\x00\x02\x00\x03\x00\x04\x00\x05\x00\x06\x00\x07\x00\x08\x00\t\x00\n\x00\x0b\x00\x0c\x00\r\x00\x0e\x00\x0f\x00\x10\x00\x11\x00\x12\x00\x13\x00\x14\x00\x15\x00\x16\x00\x17\x00\x18\x00\x19\x00\x1a\x00\x1b\x00/\x000\x001\x002\x003\x004\x005\x006\x007\x008\x009\x00:\x00;\x00<\x00=\x00>\x00?\x00@\x00A\x00B\x00C\x00D\x00E\x00F\x00g\x00h\x00i\x00j\x00k\x00l\x00m\x00\x84\x00\x85\x00\x86\x00\x87\x00\x88\x00\x89\x00\x96\x00\x97\x00\x98\x00\x99\x00\x9a\x00']
Bad pipe message: %s [b"\x9c\x00\x9d\x00\x9e\x00\x9f\x00\xa0\x00\xa1\x00\xa2\x00\xa3\x00\xa4\x00\xa5\x00\xa6\x00\xa7\x00\xba\x00\xbb\x00\xbc\x00\xbd\x00\xbe\x00\xbf\x00\xc0\x00\xc1\x00\xc2\x00\xc3\x00\xc4\x00\xc5\x13\x01\x13\x02\x13\x03\x13\x04\x13\x05\xc0\x01\xc0\x02\xc0\x03\xc0\x04\xc0\x05\xc0\x06\xc0\x07\xc0\x08\xc0\t\xc0\n\xc0\x0b\xc0\x0c\xc0\r\xc0\x0e\xc0\x0f\xc0\x10\xc0\x11\xc0\x12\xc0\x13\xc0\x14\xc0\x15\xc0\x16\xc0\x17\xc